In [1]:
import os

In [2]:
%pwd

'e:\\data science\\Deep_learning_project\\Disease_classification_project\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'e:\\data science\\Deep_learning_project\\Disease_classification_project'

In [5]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class PrepareCallbacksConfig:
    root_dir: Path
    tensorboard_root_log_dir: Path
    checkpoint_model_filepath: Path

In [7]:
from cnnclassifier.constants import *
from cnnclassifier.utils.common import read_yaml, create_directories

In [35]:
class ConfigurationManager:
    def __init__(self,
                 config_filepath = CONFIG_FILE_PATH,
                 params_filepath = PARAMS_FILE_PATH ):
        
        self.config = read_yaml(CONFIG_FILE_PATH)
        self.params = read_yaml(PARAMS_FILE_PATH)

        create_directories([self.config.artifact_root])

    def get_prepare_callbacks_config(self)->PrepareCallbacksConfig:
       
        create_directories([
            Path(self.config.prepare_callbacks.checkpoint_model_filepath),
            Path(self.config.prepare_callbacks.tensorboard_root_log_dir)
        ])

        prepare_callback_config = PrepareCallbacksConfig(
            root_dir = Path(self.config.prepare_callbacks.root_dir),
            tensorboard_root_log_dir=Path(self.config.prepare_callbacks.tensorboard_root_log_dir),
            checkpoint_model_filepath=Path(self.config.prepare_callbacks.checkpoint_model_filepath)
        )

        return prepare_callback_config

    

        

In [36]:
import os
import urllib.request as request
from zipfile import ZipFile
import tensorflow as tf
import time

In [37]:
class PrepareCallBack:

    def __init__(self,config:PrepareCallbacksConfig):

        self.config = config

    @property
    def _create_tb_callbacks(self):
        timestemp = time.strftime("%Y-%m-%d-%H-%H-%S")

        tb_running_log_dir = os.path.join(self.config.tensorboard_root_log_dir,
                                          f"tb_log_dir {timestemp}")
        return tf.keras.callbacks.TensorBoard(log_dir = tb_running_log_dir)

    @property
    def _create_ckpt_callbacks(self):
        return tf.keras.callbacks.ModelCheckpoint(
            filepath=self.config.checkpoint_model_filepath,
            save_best_only=True)

    @property
    def _create_es_callbacks(self):
        return tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=5,
            restore_best_weights=True)

    @property
    def _create_lr_callbacks(self):
        return tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.2,
            patience=3,
            min_lr=1e-6)
    
    def get_tb_ckpt_callbacks(self):
        return [
            self._create_tb_callbacks,
            self._create_ckpt_callbacks,
            self._create_es_callbacks,
            self._create_lr_callbacks
        ]


In [38]:
try: 
    config = ConfigurationManager()
    prepare_callbacks_config = config.get_prepare_callbacks_config()
    prepare_callbacks = PrepareCallBack(config= prepare_callbacks_config)
    callbacks_list = prepare_callbacks.get_tb_ckpt_callbacks()

except Exception as e:
    raise e


Loading YAML file: config\config.yaml
CONTENT:
{'artifact_root': 'artifacts', 'data_ingestion': {'root_dir': 'artifacts/data_ingestion', 'source_URL': 'https://raw.githubusercontent.com/Vasunavadiya90/data_zip/main/Chicken-fecal-images.zip', 'local_data_file': 'artifacts/data_ingestion/data.zip', 'unzip_dir': 'artifacts/data_ingestion'}, 'prepare_base_model': {'root_dir': 'artifacts/prepare_base_model', 'base_model_path': 'artifacts/prepare_base_model/base_model.h5', 'updated_base_model_path': 'artifacts/prepare_base_model/base_model_updated.h5'}, 'prepare_callbacks': {'root_dir': 'artifacts/prepare_callbacks', 'tensorboard_root_log_dir': 'artifacts/prepare_callbacks/tensorboard_log_dir', 'checkpoint_model_filepath': 'artifacts/prepare_callbacks/checkpoint_dir/model.h5'}}
TYPE:
<class 'dict'>
YAML loaded successfully: config\config.yaml

Loading YAML file: params.yaml
CONTENT:
{'AUGMENTATION': True, 'IAMGE_SIZE': [224, 224, 3], 'BATCH_SIZE': 16, 'INCLUDE_TOP': False, 'EPOCHS': 1, 'CLA

In [39]:
import time
timestamp = time.strftime("%Y-%m-%d-%H-%M-%S")
f"tb_logs_at_{timestamp}"


'tb_logs_at_2026-05-29-11-49-13'